## Imports and helper functions

In [ ]:
import glob
import pprint
import struct
from array import array
from collections.abc import Iterable
from dataclasses import dataclass, field
from enum import IntEnum
from pathlib import Path
from typing import BinaryIO

# Please install PIL or Pillow:
from PIL.ImagePalette import ImagePalette
from PIL import Image, ImageSequence

from IPython.display import display

## Jetpack classes

In [ ]:
from jetpyck.level import *

In [ ]:
for filename in sorted(glob.iglob("/home/denilson/Games/*/JETLEV.DAT")):
    print(filename)
    with open(filename, "rb") as f:
        pack = JetpackLevelPack.unpack(f)
    for i, lvl in enumerate(pack.levels):
        print("{:>3}: {}".format(i + 1, lvl.description.decode("ascii")))

## Tilemap classes (and functions) (i.e. gfx-related)

In [ ]:
class JetpackEditorTileset:
    def __init__(self, filename):
        self.filename = filename
        self.image = Image.open(self.filename)

    def __repr__(self) -> str:
        return 'JetpackEditorTileset({!r})'.format(self.filename)

    def _get_tile_box(self, id: int) -> tuple[int]:
        if not 0 <= id < 120:
            raise ValueError('Invalid tile {!r}'.format(id))

        tiles_per_row = 20
        x = id % tiles_per_row
        y = id // tiles_per_row
        w = h = 12
        dx = dy = 16

        px = 2 + x * dx
        py = 96 + y * dy
        return (px, py, px + w, py + h)

    def get_tile(self, id: int) -> Image.Image:
        box = self._get_tile_box(id)
        return self.image.crop(box)

    def get_tiles(self, id: int) -> list[Image.Image]:
        box = self._get_tile_box(id)
        return ImageSequence.all_frames(self.image, lambda image: image.crop(box))

In [ ]:
def image_from_level(level: JetpackLevel, tileset: JetpackEditorTileset) -> Image.Image:
    w = h = 12
    output = Image.new(tileset.image.mode, (w * level.tilemap.width, h * level.tilemap.height))
    for (x, y, tile) in level.tilemap.items():
        output.paste(tileset.get_tile(tile), (x * w, y * h))

    # TODO:
    # - Animation frames
    # - Add enemies
    # - Add door and player
    # - Separate layers
    # - Proper background/foreground, including transparency
    return output

## Exploration

In [ ]:
foo = JetpackLevel()
pprint.pprint(foo)

In [ ]:
pathname = Path('./levelpacks/xmasjetpack/XMAS0.JET')
with open(pathname, 'rb') as f:
    foo = JetpackLevel.unpack(f, pathname.name)
    pprint.pprint(foo)

In [ ]:
list(foo.tilemap.rows())

In [ ]:
list(foo.tilemap.items())

In [ ]:
with Image.open('./Editor-tileset.webp') as tileset:
    print(tileset.n_frames)

In [ ]:
tileset

In [ ]:

tileset.crop((2, 96, 2+12, 96+12))

In [ ]:
tileset = JetpackEditorTileset('./Editor-tileset.webp')
tileset

In [ ]:
for tile in range(0, 120):
    display(tileset.get_tile(tile))

In [ ]:
for im in tileset.get_tiles(2):
    display(im)

In [ ]:
image_from_level(foo, tileset)

In [ ]:
tileset.get_tile(4)

In [ ]:
tileset.get_tile(4).reduce(12).getpixel((0,0))

In [ ]:
tileset.get_tile(4).resize((1, 1), resample=Image.Resampling.BILINEAR).getpixel((0,0))

In [ ]:
tileset.get_tile(4).resize((1, 1), resample=Image.Resampling.BICUBIC).getpixel((0,0))

In [ ]:
tileset.get_tile(4).resize((1, 1), resample=Image.Resampling.LANCZOS).getpixel((0,0))

In [ ]:
print("u32 tileset_palette[256] = {")
for i in range(120):
    (r, g, b, a) = tileset.get_tile(i).resize((1, 1), resample=Image.Resampling.BICUBIC).getpixel((0,0))
    print("    0x{:02X}{:02X}{:02X}{:02X}, // {}".format(a, r, g, b, i))
for i in range(120, 256):
    print("    0x{:02X}{:02X}{:02X}{:02X}, // {}".format(0, 0, 0, 0, i))
print("};")